# Open API를 활용한 네이버 뉴스 검색

## 1. 애플리케이션 등록
https://developers.naver.com/apps/#/register

## 2. 환경변수 관리
- 등록된 애플리케이션 페이지에서 제공되는 Client ID, Secret은 절대(Never) 외부로 노출되면 안 된다.
- dotenv (.env)를 통해서 관리
    - `pip install python-dotenv` 패키지 설치
    - .gitignore 파일에 .env 무시하는 구문 추가

In [2]:
!python -m pip install python-dotenv

### 프로젝트 하위 폴더에 .env 파일을 만들어 아래 값 입력
```
NAVER_CLIENT_ID={{Client Id}}
NAVER_CLIENT_SECRET={{Client Secret}}
```
- 네이버 개발자센터에 등록된 애플리케이션에서 확인 가능

In [17]:
# .env 파일을 로드해서 환경변수로 등록
from dotenv import load_dotenv

load_dotenv() # .env 파일을 읽어와 '자동으로 환경 변수로 등록'
# 읽어오기 성공 시 True, 실패 시 False 반환

True

In [9]:
from pathlib import Path
import os
from dotenv import load_dotenv, dotenv_values, find_dotenv

env_path = Path.cwd() / ".env"

print("현재 Jupyter 작업 폴더:", Path.cwd())
print(".env 경로:", env_path)
print(".env 존재 여부:", env_path.exists())
print("find_dotenv(usecwd=True):", find_dotenv(usecwd=True))
print("PYTHON_DOTENV_DISABLED:", os.getenv("PYTHON_DOTENV_DISABLED"))

print("파싱된 .env 값:", dotenv_values(env_path))

loaded = load_dotenv(dotenv_path=env_path, override=True)

print("load_dotenv 결과:", loaded)

현재 Jupyter 작업 폴더: C:\SKN_33th_project\04_data_collection
.env 경로: C:\SKN_33th_project\04_data_collection\.env
.env 존재 여부: True
find_dotenv(usecwd=True): C:\SKN_33th_project\04_data_collection\.env
PYTHON_DOTENV_DISABLED: None
파싱된 .env 값: OrderedDict({'NAVER_CLIENT_ID': 'NHhn9VePF4FRYiKTV69d', 'NAVER_CLIENT_SECRET': 'ydDRErZOQR'})
load_dotenv 결과: True


In [16]:
# 환경변수에서 .env 등록한 내용 얻어오기

import os
NAVER_CLIENT_ID = os.getenv('NAVER_CLIENT_ID')
NAVER_CLIENT_SECRET = os.getenv('NAVER_CLIENT_SECRET')

# api key, 비밀번호를 print 하는 구문을 실수로 남겨놓으면 노출될 가능성 있음
# -> jupyter 변수 탭에서 확인

# .env 내용이 환경변수로 등록되지 않은 경우
if not NAVER_CLIENT_ID or not NAVER_CLIENT_SECRET:  # Falsy로 취금하는 경우
    raise ValueError('네이버 클라이언트 아이디 또는 시크릿이 문제임')

## 3. API 요청
- 파이썬에서 웹 요청(http)을 처리하기 위해서는 `requests` 라이브러리 필요

In [8]:
!python -m pip install requests

In [61]:
# 네이버 뉴스 검색 API 요청
import urllib.request   # 라이브러리 (urllib.request)
import socket

encText = urllib.parse.quote('인공지능')    # url encoding 작업
# 한글 -> %시작하는 문자로 변경

url = f'https://openapi.naver.com/v1/search/news.json?query={encText}&display=10&sort=date'
    # query(쿼리 스트링): 전달할 값(파라미터)이 작성된 문자열

# 요청 객체 생성
request = urllib.request.Request(url)   # 라이브러리에 있는 Request 클래스
                                        # --> 인스턴스가 생성 됨

# API 인증 정보를 요청 헤더에 추가
request.add_header("X-Naver-Client-Id", NAVER_CLIENT_ID)
request.add_header("X-Naver-Client-Secret", NAVER_CLIENT_SECRET)

try:
    with urllib.request.urlopen(request, timeout=10) as response:
        # 지정된 주소로 요청 -> '결과를 response로' 전달 받음
        # 단, 요청 대기 시간이 10초를 초과하면 중지
        response_code = response.getcode() # HTTP '응답 상태' 코드
                                         # 200 -> 성공,
        # 응답 본문 확인 (bytes -> UTF-8로 변환)
        response_body = response.read().decode('utf-8')

        print("response_code:", response_code)
        print("response_body:", response_body)
        # 응답 본문이 JSON(str 타입) 형태
        # -> 이용이 필요할 경우 parsing 작업 필수

except socket.timeout:
    print("요청 시간 10초 초과")

response_code: 200
response_body: {
	"lastBuildDate":"Mon, 15 Jun 2026 11:35:04 +0900",
	"total":4045703,
	"start":1,
	"display":10,
	"items":[
		{
			"title":"현대로템, 프랑스서 AI 대드론 방호체계 첫선...글로벌 시장 공략",
			"originallink":"https:\/\/www.womentimes.co.kr\/news\/articleView.html?idxno=103642",
			"link":"https:\/\/www.womentimes.co.kr\/news\/articleView.html?idxno=103642",
			"description":"우먼타임스 = 장준형 기자 현대로템이 세계 최대 방산 전시회에서 <b>인공지능<\/b>(AI) 기반 첨단 방호 기술과 지상무기체계 라인업을 선보인다. 현대로템은 15일(현지 시간)부터 19일까지 프랑스 파리에서 열리는 '유로사토리... ",
			"pubDate":"Mon, 15 Jun 2026 11:34:00 +0900"
		},
		{
			"title":"경과원-경기도, 스타트업 아카데미 마케팅·글로벌 진출 과정 교육생 모...",
			"originallink":"https:\/\/www.ajunews.com\/view\/20260615112808371",
			"link":"https:\/\/www.ajunews.com\/view\/20260615112808371",
			"description":"디지털 마케팅 실행 과정에서는 <b>인공지능<\/b>을 활용한 상세페이지 카피라이팅, 브랜드 일관성 관리, B2B 마케팅 자산화 등 창업기업이 실제 판매와 홍보 과정에서 활용할 수 있는 내용을 포함한다. 초기 기업이 제한된... ",
			"pubDate":"Mon, 15 Jun 2026 11:34:00 +0900"
		},
		{
			"title":"스마일게이트, 인디 창

In [82]:
# requests 객체를 이용한 요청 (더 쉬움)
import requests
from pprint import pprint
# 출력 시 공백문자를 통해 가독성 있게 출력함

url = 'https://openapi.naver.com/v1/search/news.json'

# Header, Body에 전달할 값을 dict 생성
headers = {
    'X-Naver-Client-Id': NAVER_CLIENT_ID,
    'X-Naver-Client-Secret': NAVER_CLIENT_SECRET
}

params = {
    'query' : '인공지능',
    'display' : '10',
    'start' : 1,
    'sort' : 'date'
}


try:
    # GET Method == 조회 요청
    response = requests.get(
        url,  # 인자가 너무 많아 url을 제외한 나머지는 키워드 인자로 값을 넣어줌
        headers=headers,
        params=params,  # dict -> 쿼리 스트링 변환 (url encoding)
        timeout=10
    )

    # HTTP 상태 코드가 400, 500번대인 경우 예외 발생
    response.raise_for_status() # raise: '에러를 던지겠다'

    response_code = response.status_code    # 상태 코드
    data = response.json() # 응답 데이터 (json) -> dict 변환

    print("response_code:", response_code)
    # pprint(data)
    pprint(data['items'][0]['description'])

except requests.exceptions.Timeout:
    print("요청 시간 10초 초과")
except ValueError:
    print("응답 데이터가 JSON 형식이 아닙니다")

response_code: 200
('주로 <b>인공지능</b>(AI), 블록체인, 로봇, 방위산업 등에 투자를 하고 있다. 단순 재무적 투자 외 마케팅, 홍보, 영업 등 '
 '업무 전반에 전문가들이 투자 회사를 지원하는 점이 특징이다. 현재 이 업체가 운용하는 자산... ')


In [87]:
# requests 객체를 이용한 요청 (더 쉬움)
import requests
from pprint import pprint
# 출력 시 공백문자를 통해 가독성 있게 출력함

## 다수의 공공데이터: xml 등의 형태로 제공 not json.
url = 'https://openapi.naver.com/v1/search/news.xml'

# Header, Body에 전달할 값을 dict 생성
headers = {
    'X-Naver-Client-Id': NAVER_CLIENT_ID,
    'X-Naver-Client-Secret': NAVER_CLIENT_SECRET
}

params = {
    'query' : '인공지능',
    'display' : '10',
    'start' : 1,
    'sort' : 'date'
}


try:
    # GET Method == 조회 요청
    response = requests.get(
        url,  # 인자가 너무 많아 url을 제외한 나머지는 키워드 인자로 값을 넣어줌
        headers=headers,
        params=params,  # dict -> 쿼리 스트링 변환 (url encoding)
        timeout=10
    )

    # HTTP 상태 코드가 400, 500번대인 경우 예외 발생
    response.raise_for_status() # raise: '에러를 던지겠다'

    response_code = response.status_code    # 상태 코드
    data = response.content # 응답 문자열 (xml)

    # ************************************
    # XML 해석 파이썬 기본 내장 모듈
    # -> xml.etree.ElementTree
    # ************************************

    # XML 파일로 저장
    with open('response.xml', 'wb') as f:
        f.write(data)

    print("response.xml 저장 완료")

except requests.exceptions.Timeout:
    print("요청 시간 10초 초과")

response.xml 저장 완료
